In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, concat_ws, lower, year

# ----------------------------------------------------------
# 1. Khởi tạo SparkSession với cấu hình MinIO (S3A)
# ----------------------------------------------------------
spark = (
    SparkSession.builder
    .appName("Transfermarkt ETL - Bronze to Silver")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "admin12345")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)

print("Spark session started successfully.")

# ----------------------------------------------------------
# 2. Đọc dữ liệu từ tầng Bronze (MinIO bucket)
# ----------------------------------------------------------
bronze_path = "s3a://transfermarkt-bronze/players.csv"

df_players = spark.read.csv(bronze_path, header=True, inferSchema=True)
print("Đã đọc dữ liệu Bronze từ:", bronze_path)
df_players.printSchema()
print("Số dòng ban đầu:", df_players.count())

# ----------------------------------------------------------
# 3. Làm sạch và chuẩn hóa dữ liệu
# ----------------------------------------------------------
# - Gộp họ và tên thành 'name'
# - Loại bỏ bản ghi thiếu tên hoặc ngày sinh
# - Chuẩn hóa định dạng tên về chữ thường
# - Tính tuổi tại mùa giải cuối cùng (last_season - năm sinh)
df_players_clean = (
    df_players
    .withColumn("name", concat_ws(" ", trim(col("first_name")), trim(col("last_name"))))
    .dropna(subset=["name", "date_of_birth", "last_season"])
    .withColumn("name", trim(lower(col("name"))))
    .withColumn("age", col("last_season") - year(col("date_of_birth")))
)

print("Hoàn tất bước làm sạch dữ liệu.")
df_players_clean.printSchema()
df_players_clean.show(5, truncate=False)

# ----------------------------------------------------------
# 4. Ghi dữ liệu sang tầng Silver (định dạng Parquet)
# ----------------------------------------------------------
silver_path = "s3a://transfermarkt-silver/players"

df_players_clean.write.mode("overwrite").parquet(silver_path)
print("Dữ liệu Silver đã được ghi tại:", silver_path)

# ----------------------------------------------------------
# 5. Kiểm tra dữ liệu sau khi ghi
# ----------------------------------------------------------
df_check = spark.read.parquet(silver_path)
print("Số dòng trong Silver:", df_check.count())
df_check.select("name", "age", "position", "current_club_name").show(10, truncate=False)
